In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [16]:
df = pd.read_csv("/content/after_feature_selection.csv")

In [17]:
df1 =  pd.read_csv("/content/after_feature_engineering.csv")

In [21]:
final_df = df1[['JobTitle', 'Company Size', 'company_Industry', 'Country',
       'Remote Type', 'Exp. Level', 'Years_Experience', 'Education  Level',
       'Skills - Python', 'Skills - SQL', 'Skills - ML', 'Skills-DeepLearning',
       'skills_Cloud', 'Posting Month', 'posting_Year', 'Hiring Urgency',
       'Total_Skills', 'advanced_skill_score', 'ai_specialist',
       'exp_per_skill', 'experience_intensity', 'ml_cloud_combination',
       'job_title_popularity', 'industry_frequency', 'premium_candidate',
       'Salary (USD)']]

In [26]:
final_df.to_csv("final_df.csv",index=False)

In [25]:
final_df.shape

(10025, 26)

In [22]:
final_df.head(1)

,JobTitle,Company Size,company_Industry,Country,Remote Type,Exp. Level,Years_Experience,Education Level,Skills - Python,Skills - SQL,...,Total_Skills,advanced_skill_score,ai_specialist,exp_per_skill,experience_intensity,ml_cloud_combination,job_title_popularity,industry_frequency,premium_candidate,Salary (USD)
0,Data Analyst,Start-up,Finance,Unknown,Hybrid,Junior,4.0,Bachelor's,1,1,...,2,3,0,2.0,12.0,0,1626,1595,0,61004.0


In [24]:
final_df["JobTitle"].unique()

array(['Data Analyst', 'Data Scientist', 'Machine Learning Engineer',
       'AI Engineer', 'Business Analyst', 'Data Engineer', 'Unknown'],
      dtype=object)

In [27]:
df = final_df

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10025 entries, 0 to 10024
Data columns (total 26 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   JobTitle              10025 non-null  object 
 1   Company Size          10025 non-null  object 
 2   company_Industry      10025 non-null  object 
 3   Country               10025 non-null  object 
 4   Remote Type           10025 non-null  object 
 5   Exp. Level            10025 non-null  object 
 6   Years_Experience      10025 non-null  float64
 7   Education  Level      10025 non-null  object 
 8   Skills - Python       10025 non-null  int64  
 9   Skills - SQL          10025 non-null  int64  
 10  Skills - ML           10025 non-null  int64  
 11  Skills-DeepLearning   10025 non-null  int64  
 12  skills_Cloud          10025 non-null  int64  
 13  Posting Month         10025 non-null  int64  
 14  posting_Year          10025 non-null  int64  
 15  Hiring Urgency     

In [46]:
X = df.drop(columns=['Salary (USD)'])
y = df['Salary (USD)']

# Replace infinite values with NaN in numerical columns
for col in num_cols:
    if col in X.columns:
        X[col] = X[col].replace([np.inf, -np.inf], np.nan)

In [47]:
y_transformed = np.log1p(y)

In [49]:
col_to_encode = ['JobTitle','company_Industry','Country','Remote Type']
num_cols = ['Years_Experience','Total_Skills','advanced_skill_score','ai_specialist','exp_per_skill','experience_intensity',
'ml_cloud_combination','job_title_popularity','industry_frequency','premium_candidate']
ordinal_cols = ['Company Size','Exp. Level','Education  Level','Hiring Urgency']

total_cat_cols = ['JobTitle','company_Industry','Country','Remote Type','Company Size','Exp. Level','Education  Level','Hiring Urgency']

In [32]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, OrdinalEncoder

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

from sklearn.decomposition import PCA

In [50]:
preprocessor = ColumnTransformer(
    transformers = [
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='mean')), ('scaler', StandardScaler())]), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown="ignore"), col_to_encode),
        ('ordinal', OrdinalEncoder(), ordinal_cols)
    ],
    remainder = 'passthrough'
)

In [43]:
model_dict = {
    'linear_reg': LinearRegression(),
    'svr': SVR(),
    'ridge': Ridge(),
    'LASSO': Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest': RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'knn' : KNeighborsRegressor(),
    'xgboost':XGBRegressor()
}

In [44]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    # appending cv
    output.append(scores.mean())

    # train and split
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    # train data
    pipeline.fit(X_train,y_train)

    # prediction
    y_pred = pipeline.predict(X_test)

    # mean_absolute_errror
    output.append(mean_absolute_error(y_test,y_pred))

    return output

In [51]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [52]:
model_df = pd.DataFrame(model_output, columns=['Model Name','Cross Validation Mean(r2)','MAE'])

In [53]:
model_df.sort_values(by='Cross Validation Mean(r2)',ascending=False)

,Model Name,Cross Validation Mean(r2),MAE
7,gradient boosting,0.765469,0.102087
5,random forest,0.746431,0.105305
9,xgboost,0.730121,0.110225
6,extra trees,0.709859,0.109426
2,ridge,0.595468,0.149342
0,linear_reg,0.595463,0.149317
4,decision tree,0.490411,0.144737
8,knn,0.463243,0.173720
1,svr,0.011787,0.237845
3,LASSO,-0.000614,0.241002


**HyperParameter Tunning**

In [54]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 11.5 MB/s eta 0:00:00


In [55]:
y_transformed = np.log1p(y)

In [56]:
# train and split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_transformed,
    test_size=0.2,
    random_state=42
)

In [57]:
import optuna

def objective(trial):

    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'min_samples_split': trial.suggest_int('min_samples_split',3,5,6),
        'alpha':0.9,
        'min_samples_leaf':trial.suggest_int('min_samples_leaf',3,4,2),
        'random_state': 42
    }

    model = GradientBoostingRegressor(**params)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    score = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring='r2'
    ).mean()

    return score

In [59]:
study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=30)

[I 2026-05-22 15:05:58,486] A new study created in memory with name: no-name-cc7edfcb-a117-4da0-9d47-f75126697805
/tmp/ipykernel_2587/3698674088.py:10: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'min_samples_split': trial.suggest_int('min_samples_split',3,5,6),
/tmp/ipykernel_2587/3698674088.py:10: UserWarning: The distribution is specified by [3, 5] and step=6, but the range is not divisible by `step`. It will be replaced with [3, 3].
  'min_samples_split': trial.suggest_int('min_samples_split',3,5,6),
/tmp/ipykernel_2587/3698674088.py:12: FutureWarning: suggest_int() got {'step'} as positional ar

In [60]:
print(study.best_params)
print(study.best_value)

{'n_estimators': 146, 'max_depth': 6, 'learning_rate': 0.034169408523184605, 'subsample': 0.7271957508879802, 'min_samples_split': 3, 'min_samples_leaf': 3}
0.7608930251295583


In [61]:
model1 = GradientBoostingRegressor(
    **study.best_params,
    random_state = 42
)

pipeline1 = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model1)
])

pipeline1.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Years_Experience',
                                                   'Total_Skills',
                                                   'advanced_skill_score',
                                                   'ai_specialist',
                                                   'exp_per_skill',
                                                   'experience_intensity',
                                                   'ml_cloud_combination',
                                                   'job_title_popularity',
                                                   'industry_frequency',
                                                   '...
                                                                handle_unknown='ignore'),
                                                  ['JobTitle',
                                                   'company_Industry',
                                                   'Country', 'Remote Type']),
                                                 ('ordinal', OrdinalEncoder(),
                                                  ['Company Size', 'Exp. Level',
                                                   'Education  Level',
                                                   'Hiring Urgency'])])),
                ('model',
                 GradientBoostingRegressor(learning_rate=0.034169408523184605,
                                           max_depth=6, min_samples_leaf=3,
                                           min_samples_split=3,
                                           n_estimators=146, random_state=42,
                                           subsample=0.7271957508879802))])

In [62]:
y_pred = pipeline1.predict(X_test)

In [63]:
from sklearn.metrics import r2_score

print("Gradient boosting R2 score", r2_score(y_test, y_pred))

Gradient boosting R2 score 0.77227171223041


In [64]:
import joblib

joblib.dump(pipeline1, 'salary_prediction_pipeline.pkl')

['salary_prediction_pipeline.pkl']

In [65]:
import joblib

joblib.dump(model1, 'salary_prediction_model.pkl')

['salary_prediction_model.pkl']

**Random Forest Regressor**

In [ ]:
def objective(trial):

    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 200, 500),
        'max_depth': trial.suggest_int('max_depth', 3,6, 10),
        'min_samples_split': trial.suggest_int('min_samples_split',3,5,6),
        'min_samples_leaf':trial.suggest_int('min_samples_leaf',3,4,2),
        'random_state': 42
    }

    model = RandomForestRegressor(**params)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    score = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring='r2'
    ).mean()

    return score

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

[I 2026-05-19 16:14:44,717] A new study created in memory with name: no-name-9df26110-3464-46b8-9813-30fe03b2a5df
/tmp/ipykernel_5527/3628030789.py:4: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'n_estimators': trial.suggest_int('n_estimators', 100, 200, 500),
/tmp/ipykernel_5527/3628030789.py:4: UserWarning: The distribution is specified by [100, 200] and step=500, but the range is not divisible by `step`. It will be replaced with [100, 100].
  'n_estimators': trial.suggest_int('n_estimators', 100, 200, 500),
/tmp/ipykernel_5527/3628030789.py:5: FutureWarning: suggest_int() got {'step'} as position

In [ ]:
print(study.best_params)
print(study.best_value)

{'n_estimators': 100, 'max_depth': 3, 'min_samples_split': 3, 'min_samples_leaf': 3}
0.5671406874727536
